In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mxlpy import Simulator, scan

from mxlmodels import get_zhu_2009

In [ ]:
res = Simulator(get_zhu_2009()).simulate(650).get_result().unwrap_or_err()

y_lim_max = [4, 3, 3, 8, 0.1]

fig, axes = plt.subplots(
    3,
    2,
    figsize=(9, 10),
)
axes = axes.flatten()
for i, col in enumerate(["RuBP", "PGA", "DPGA", "GAP", "Ru5P"]):
    ax = axes[i]
    ax.plot(res.variables.index, res.variables[col], label=col)
    ax.set_xlim(left=0)
    ax.set_ylim(0, y_lim_max[i])
    ax.set_xlabel("time (s)")
    ax.set_ylabel("Concentration (mM)")
    ax.legend()

axes[5].plot(
    res.fluxes.index,
    res.fluxes["v1"]
    * 33.33,  # Here the carbon fixation is calculated by multiplying the flux with 33.33 (see Zhu et al. 2009)
    label="A",
)
axes[5].set_xlim(left=0)
axes[5].set_ylim(35, 60)
axes[5].set_xlabel("time in s")
axes[5].set_ylabel("A (umol m-2 s-1)")
axes[5].legend()

In [ ]:
V1_max = 3.78
V2_max = 11.75
V3_max = 5.04
V4_max = 3.05  # estimate
V5_max = 3
V6_max = 0.1  # estimate
V13_max = 8

V_max_variable = [V1_max, V3_max, V4_max, V13_max, V2_max, V5_max]

np_percent = np.array([50, 100, 150, 200, 250, 300])
fig, axes = plt.subplots(3, 2, figsize=(10, 10))
axes = axes.flatten()

for i, V_max in enumerate(
    ["V1_max", "V3_max", "V4_max", "V13_max", "V2_max", "V5_max"]
):
    res = scan.steady_state(
        get_zhu_2009(),
        to_scan=pd.DataFrame({V_max: V_max_variable[i] * np_percent / 100}),
    )
    ax = axes[i]
    ax.scatter(np_percent, res.variables.loc[:, ["RuBP"]], label=["RuBP"], marker="o")
    ax.scatter(
        np_percent,
        res.variables.loc[:, ["PGA"]],
        label=[
            "PGA",
        ],
        marker="s",
    )
    ax.scatter(
        np_percent,
        res.variables.loc[:, ["DPGA"]],
        label=[
            "DPGA",
        ],
        marker="^",
    )
    ax.scatter(
        np_percent,
        res.variables.loc[:, ["Ru5P"]],
        label=[
            "Ru5P",
        ],
        marker="d",
    )
    ax.set_xlabel(f"{V_max} (% of original value)")
    ax.set_ylabel("Concentration (mM)")
    ax.set_xlim(50, 300)

ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.show()